# 01 - Dataset Exploration & Stratified Sampling
## CSE-CIC-IDS2018 (10 file)

**Tahap:**
1. Baca seluruh dataset → distribusi populasi & komposisi persentase per attack type
2. Stratified Sampling 10% → `file_100.csv`
3. Buat subset: `file_75.csv` (75%), `file_50.csv` (50%), `file_25.csv` (25%)

**Setting:** `setting.txt` → mode=0 (local) atau mode=1 (S3)

In [ ]:
import pandas as pd
import numpy as np
import os, glob, gc, warnings
warnings.filterwarnings('ignore')

# Read setting
setting = {}
with open('setting.txt', 'r') as f:
    for line in f:
        key, val = line.strip().split('=')
        setting[key] = val

MODE = int(setting['mode'])

if MODE == 0:
    DATA_DIR = '../data/'
elif MODE == 1:
    # Download from S3 if not exists
    DATA_DIR = '../data/'
    if not os.path.exists(DATA_DIR) or len(glob.glob(os.path.join(DATA_DIR, '*.csv'))) < 10:
        print('Downloading dataset from S3...')
        os.system('aws s3 sync s3://ssh-detection-features-232032302717/datasets/CICD2018/ ../data/ --quiet')
        print('Download complete.')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f'MODE={MODE} | Data dir: {DATA_DIR}')

## 1. Membaca & Memeriksa Seluruh Dataset

In [ ]:
# List semua CSV (kecuali SSH-Bruteforce.csv yang merupakan subset extracted)
all_csv = sorted(glob.glob(os.path.join(DATA_DIR, '*2018*.csv')))
print(f'Dataset files ({len(all_csv)}):')
for f in all_csv:
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'  {os.path.basename(f):55s} ({size_mb:.1f} MB)')

In [ ]:
# Scan seluruh file untuk hitung distribusi label TANPA load semua ke memory
# Hanya baca kolom label per chunk

print('Scanning all files for label distribution...')
print('(This may take several minutes for large files)')
print()

global_label_counts = {}
total_rows = 0
file_info = []

for filepath in all_csv:
    fname = os.path.basename(filepath)
    print(f'Scanning: {fname}...', end=' ')
    
    file_rows = 0
    file_labels = {}
    
    try:
        for chunk in pd.read_csv(filepath, chunksize=100000, low_memory=False,
                                  encoding='utf-8', on_bad_lines='skip'):
            chunk.columns = chunk.columns.str.strip()
            # Find label column
            label_col = [c for c in chunk.columns if 'label' in c.lower()]
            LABEL = label_col[0] if label_col else chunk.columns[-1]
            
            # Count labels
            counts = chunk[LABEL].value_counts().to_dict()
            for label, count in counts.items():
                label = str(label).strip()
                global_label_counts[label] = global_label_counts.get(label, 0) + count
                file_labels[label] = file_labels.get(label, 0) + count
            file_rows += len(chunk)
    except Exception as e:
        print(f'ERROR: {e}')
        continue
    
    total_rows += file_rows
    file_info.append({'file': fname, 'rows': file_rows, 'labels': file_labels})
    print(f'{file_rows:,} rows | Labels: {list(file_labels.keys())}')

print(f'\n{"="*70}')
print(f'TOTAL ROWS: {total_rows:,}')
print(f'TOTAL ATTACK TYPES: {len(global_label_counts)}')
print(f'{"="*70}')

In [ ]:
# Distribusi populasi asli (persentase)
print(f'\n{"="*70}')
print(f'{"DISTRIBUSI POPULASI ASLI":^70}')
print(f'{"="*70}')
print(f'{"Label":40s} {"Count":>12s} {"Percentage":>12s}')
print(f'{"-"*70}')

sorted_labels = sorted(global_label_counts.items(), key=lambda x: x[1], reverse=True)
for label, count in sorted_labels:
    pct = count / total_rows * 100
    print(f'{label:40s} {count:>12,} {pct:>11.4f}%')

print(f'{"-"*70}')
print(f'{"TOTAL":40s} {total_rows:>12,} {100.0:>11.4f}%')
print(f'{"="*70}')

In [ ]:
# Detail per file
print(f'\n{"="*70}')
print(f'{"DETAIL PER FILE":^70}')
print(f'{"="*70}')
for info in file_info:
    print(f'\n{info["file"]} ({info["rows"]:,} rows):')
    for label, count in sorted(info['labels'].items(), key=lambda x: x[1], reverse=True):
        pct = count / info['rows'] * 100
        print(f'  {label:35s} {count:>10,} ({pct:.2f}%)')

In [ ]:
# Install dependencies (jalankan sekali)
import sys
!{sys.executable} -m pip install matplotlib seaborn scikit-learn xgboost lightgbm joblib psutil -q

## 1.5 Visualisasi Distribusi Populasi

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

In [ ]:
# === BAR CHART: Distribusi Label Populasi Asli ===
labels_sorted = [lbl for lbl, _ in sorted_labels]
counts_sorted = [cnt for _, cnt in sorted_labels]

fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.tab20(np.linspace(0, 1, len(labels_sorted)))
bars = ax.bar(range(len(labels_sorted)), counts_sorted, color=colors, edgecolor='black', linewidth=0.5)

ax.set_xticks(range(len(labels_sorted)))
ax.set_xticklabels(labels_sorted, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Jumlah Flow')
ax.set_title('Distribusi Label Populasi Asli — CSE-CIC-IDS2018', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K' if x >= 1e3 else f'{x:.0f}'))

# Tambah label count di atas bar
for bar, count in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{count:,.0f}', ha='center', va='bottom', fontsize=7, rotation=0)

plt.tight_layout()
plt.savefig('../data/bar_distribusi_label_populasi.png', bbox_inches='tight')
plt.show()
print('Saved: bar_distribusi_label_populasi.png')

In [ ]:
# === PIE CHART: Komposisi Attack Types ===
# Group into categories for clearer pie chart
attack_only = {lbl: cnt for lbl, cnt in global_label_counts.items() if lbl.lower() not in ['benign', 'normal']}
benign_count = sum(cnt for lbl, cnt in global_label_counts.items() if lbl.lower() in ['benign', 'normal'])

# Sort attacks by count
attack_sorted = sorted(attack_only.items(), key=lambda x: x[1], reverse=True)
attack_labels = [lbl for lbl, _ in attack_sorted]
attack_counts = [cnt for _, cnt in attack_sorted]

# Pie chart — semua attack types + Benign
all_pie_labels = ['Benign'] + attack_labels
all_pie_counts = [benign_count] + attack_counts

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Full composition (Benign vs Attack)
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(all_pie_labels)))
explode = [0.05 if lbl == 'Benign' else 0 for lbl in all_pie_labels]
wedges1, texts1, autotexts1 = ax1.pie(
    all_pie_counts, labels=None, autopct='%1.1f%%',
    colors=colors_pie, explode=explode, startangle=90,
    pctdistance=0.85, textprops={'fontsize': 8})
ax1.set_title('Komposisi Seluruh Traffic\n(Benign + Attack)', fontsize=11, fontweight='bold')
ax1.legend(all_pie_labels, loc='lower left', fontsize=7, bbox_to_anchor=(-0.3, 0))

# Right: Attack types only
colors_attack = plt.cm.tab20(np.linspace(0, 1, len(attack_labels)))
wedges2, texts2, autotexts2 = ax2.pie(
    attack_counts, labels=None, autopct='%1.1f%%',
    colors=colors_attack, startangle=90,
    pctdistance=0.82, textprops={'fontsize': 8})
ax2.set_title('Komposisi Attack Types Only\n(tanpa Benign)', fontsize=11, fontweight='bold')
ax2.legend(attack_labels, loc='lower left', fontsize=7, bbox_to_anchor=(-0.3, 0))

plt.tight_layout()
plt.savefig('../data/pie_komposisi_attack_types.png', bbox_inches='tight')
plt.show()
print('Saved: pie_komposisi_attack_types.png')

In [ ]:
# === COMPARISON PLOT: Original vs Sampled Percentage ===
# (Akan di-update setelah sampling selesai — placeholder yang langsung jalan setelah sampling)
# Untuk saat ini, simpan data populasi dulu
original_pct = {lbl: cnt/total_rows*100 for lbl, cnt in global_label_counts.items()}
print('Original percentages saved. Comparison plot will be generated after sampling.')
print(f'Labels: {len(original_pct)} | Total rows: {total_rows:,}')

## 2. Stratified Sampling 10% → file_100.csv

In [ ]:
# Hitung target per label (10% dari populasi asli)
SAMPLE_RATIO = 0.10
target_per_label = {}
for label, count in global_label_counts.items():
    target = max(1, int(count * SAMPLE_RATIO))  # minimal 1 sample per class
    target_per_label[label] = target

total_target = sum(target_per_label.values())
print(f'Sampling ratio: {SAMPLE_RATIO*100:.0f}%')
print(f'Target total samples: {total_target:,} (from {total_rows:,})')
print(f'\nTarget per label:')
for label, target in sorted(target_per_label.items(), key=lambda x: x[1], reverse=True):
    pct = target / total_target * 100
    print(f'  {label:35s} {target:>10,} ({pct:.4f}%)')

In [ ]:
# Stratified sampling dari semua file
print('\nPerforming stratified sampling...')
print('(Loading data per file, sampling proportionally)')

# Track how many we've sampled per label
sampled_per_label = {label: 0 for label in global_label_counts}
sampled_chunks = []

for filepath in all_csv:
    fname = os.path.basename(filepath)
    print(f'  Sampling from {fname}...', end=' ')
    
    try:
        # Read file in chunks to manage memory
        file_samples = []
        for chunk in pd.read_csv(filepath, chunksize=200000, low_memory=False,
                                  encoding='utf-8', on_bad_lines='skip'):
            chunk.columns = chunk.columns.str.strip()
            label_col = [c for c in chunk.columns if 'label' in c.lower()]
            LABEL = label_col[0] if label_col else chunk.columns[-1]
            
            # Sample proportionally per label in this chunk
            for label, group in chunk.groupby(LABEL):
                label = str(label).strip()
                # How many more do we need for this label?
                remaining = target_per_label.get(label, 0) - sampled_per_label.get(label, 0)
                if remaining <= 0:
                    continue
                # Calculate proportion to sample from this chunk
                n_sample = min(remaining, max(1, int(len(group) * SAMPLE_RATIO)))
                n_sample = min(n_sample, len(group))
                if n_sample > 0:
                    sampled = group.sample(n=n_sample, random_state=RANDOM_SEED)
                    # Standardize label column name
                    sampled = sampled.rename(columns={LABEL: 'Label'})
                    sampled['Label'] = label
                    file_samples.append(sampled)
                    sampled_per_label[label] = sampled_per_label.get(label, 0) + n_sample
        
        if file_samples:
            file_df = pd.concat(file_samples, ignore_index=True)
            sampled_chunks.append(file_df)
            print(f'{len(file_df):,} samples')
        else:
            print('0 samples')
        
        del file_samples
        gc.collect()
        
    except Exception as e:
        print(f'ERROR: {e}')
        continue

# Combine all samples
df_100 = pd.concat(sampled_chunks, ignore_index=True)
del sampled_chunks
gc.collect()

print(f'\nfile_100 total: {len(df_100):,} rows')

In [ ]:
# Verify distribution preservation
print(f'\n{"="*70}')
print(f'{"VERIFIKASI: Distribusi file_100 vs Populasi Asli":^70}')
print(f'{"="*70}')
print(f'{"Label":35s} {"Original%":>12s} {"Sampled%":>12s} {"Diff":>8s}')
print(f'{"-"*70}')

sample_dist = df_100['Label'].value_counts().to_dict()
for label, orig_count in sorted(global_label_counts.items(), key=lambda x: x[1], reverse=True):
    orig_pct = orig_count / total_rows * 100
    samp_count = sample_dist.get(label, 0)
    samp_pct = samp_count / len(df_100) * 100 if len(df_100) > 0 else 0
    diff = samp_pct - orig_pct
    print(f'{label:35s} {orig_pct:>11.4f}% {samp_pct:>11.4f}% {diff:>+7.3f}')

print(f'{"="*70}')

In [ ]:
# === COMPARISON PLOT: Original vs Sampled Percentage ===
sample_dist = df_100['Label'].value_counts().to_dict()
sampled_pct = {lbl: cnt/len(df_100)*100 for lbl, cnt in sample_dist.items()}

# Prepare data
comp_labels = [lbl for lbl, _ in sorted(global_label_counts.items(), key=lambda x: x[1], reverse=True)]
orig_pcts = [original_pct[lbl] for lbl in comp_labels]
samp_pcts = [sampled_pct.get(lbl, 0) for lbl in comp_labels]

x = np.arange(len(comp_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, orig_pcts, width, label='Original Population', color='steelblue', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, samp_pcts, width, label='Sampled (10%)', color='coral', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Attack Type')
ax.set_ylabel('Percentage (%)')
ax.set_title('Perbandingan Distribusi: Original Population vs Stratified Sample (10%)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comp_labels, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=11)

# Add percentage labels on bars
for bar in bars1:
    h = bar.get_height()
    if h > 0.5:
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}%', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    h = bar.get_height()
    if h > 0.5:
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}%', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../data/comparison_original_vs_sampled.png', bbox_inches='tight')
plt.show()

# Print deviation summary
print(f'\nMax deviation: {max(abs(o-s) for o,s in zip(orig_pcts, samp_pcts)):.4f}%')
print(f'Mean deviation: {np.mean([abs(o-s) for o,s in zip(orig_pcts, samp_pcts)]):.4f}%')
print('Saved: comparison_original_vs_sampled.png')

## 3. Generate Subset Files

In [ ]:
# Shuffle file_100
df_100 = df_100.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Save file_100.csv (10% of original = 100% of our working set)
output_dir = '../data/'
df_100.to_csv(os.path.join(output_dir, 'file_100.csv'), index=False)
print(f'Saved: file_100.csv ({len(df_100):,} rows)')

# file_75.csv = 75% of file_100 (stratified)
df_75 = df_100.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(frac=0.75, random_state=RANDOM_SEED)
).reset_index(drop=True)
df_75.to_csv(os.path.join(output_dir, 'file_75.csv'), index=False)
print(f'Saved: file_75.csv ({len(df_75):,} rows)')

# file_50.csv = 50% of file_100 (stratified)
df_50 = df_100.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(frac=0.50, random_state=RANDOM_SEED)
).reset_index(drop=True)
df_50.to_csv(os.path.join(output_dir, 'file_50.csv'), index=False)
print(f'Saved: file_50.csv ({len(df_50):,} rows)')

# file_25.csv = 50% of file_50 (= 25% of file_100, stratified)
df_25 = df_50.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(frac=0.50, random_state=RANDOM_SEED)
).reset_index(drop=True)
df_25.to_csv(os.path.join(output_dir, 'file_25.csv'), index=False)
print(f'Saved: file_25.csv ({len(df_25):,} rows)')

print(f'\n=== Summary ===')
print(f'file_100.csv: {len(df_100):>10,} rows (10% of total {total_rows:,})')
print(f'file_75.csv:  {len(df_75):>10,} rows (75% of file_100)')
print(f'file_50.csv:  {len(df_50):>10,} rows (50% of file_100)')
print(f'file_25.csv:  {len(df_25):>10,} rows (25% of file_100)')

In [ ]:
# Final verification
print(f'\n{"="*70}')
print(f'{"RINGKASAN AKHIR":^70}')
print(f'{"="*70}')
print(f'Total data asli: {total_rows:,} rows')
print(f'Jumlah attack types: {len(global_label_counts)}')
print(f'Sampling method: Stratified ({SAMPLE_RATIO*100:.0f}%)')
print(f'\nOutput files:')
for fname in ['file_100.csv', 'file_75.csv', 'file_50.csv', 'file_25.csv']:
    fpath = os.path.join(output_dir, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'  {fname:15s} ({size_mb:.1f} MB)')
print(f'{"="*70}')
print('\nDONE! Files ready for model training.')